In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import os
import time

In [ ]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
MODEL_NAME = "Logistic Regression"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.real_heldout("2d")
feature_names = EXP.feature_names
X_train_scaled = EXP.X_train
X_test_scaled = EXP.X_test
y_train = EXP.y_train
y_test = EXP.y_test


In [5]:
# Training the model
print("\nTraining the model...")

model = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)
start_time = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time


Training the model...


In [6]:
# Model evaluation
print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 77.89%
F1 Score: 0.6825

Classification Report:
              precision    recall  f1-score   support

           0       0.78      1.00      0.88    318260
           1       0.00      0.00      0.00     29166
           2       0.08      0.00      0.00     35386
           3       0.43      0.00      0.00     25612

    accuracy                           0.78    408424
   macro avg       0.32      0.25      0.22    408424
weighted avg       0.64      0.78      0.68    408424



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="Regular",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
